# Event — brapi-py

Interactive notebook for exploring the BrAPI **Event** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Uncomment to install / upgrade brapi-py
# %pip install brapi-py
import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient

# ── Edit these ────────────────────────────────────────────────────────────
BASE_URL       = 'https://brapi.example.com'
TOKEN_ENDPOINT = 'https://auth.example.com/token'
CLIENT_ID      = 'my-client'
CLIENT_SECRET  = 'my-secret'
# ─────────────────────────────────────────────────────────────────────────

client = BrapiClient(
    base_url=BASE_URL,
    token_endpoint=TOKEN_ENDPOINT,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)
print('Client ready →', client)

## 3 — List  (`GET /events`)

Simpler GET endpoint — same filter state, mapped to query-string params.
Useful when the server does not support the search endpoint, or for quick lookups.

In [ ]:
# GET /events — list records
df = (
    client.event
    .by_study_dbid(['cf6c4bd4'])
    .list()
    .to_df()
)
print(f'{len(df)} records')
df.head()

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.event
    .by_study_dbid(['cf6c4bd4'])
    .list()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'eventDescription'
if 'eventDescription' in df.columns:
    df['eventDescription'].value_counts().head(10)
else:
    print('Column not present in this result set')

In [ ]:
import json, pandas as pd

# Explode the 'eventDateRange' JSON column into separate rows
col = 'eventDateRange'
if col in df.columns:
    id_col = ''
    exploded = (
        df.filter(items=[id_col, col])
        .dropna(subset=[col])
        .assign(**{col: lambda d: d[col].apply(json.loads)})
        .explode(col)
    )
    exploded.head(10)
else:
    print(f'Column {col!r} not present in this result set')

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')